# Sprite LoRA Training (Kaggle)
Trains a LoRA adapter on your sprite dataset using **ai-toolkit**.

**Default: SD 1.5** (fits free Kaggle T4 15GB). FLUX.1-dev needs >=24GB VRAM.
To use FLUX, edit MODEL_ID in Step 3 and run on a paid GPU (A100).

In [ ]:
# Step 1: Install Dependencies
!pip install -U -q 'numpy<2.0' 'pandas<3.0' # Force compatible versions
!pip install -q torch torchvision accelerate transformers diffusers peft bitsandbytes safetensors sentencepiece rembg
!git clone https://github.com/ostris/ai-toolkit.git /kaggle/working/ai-toolkit
%cd /kaggle/working/ai-toolkit
!git submodule update --init --recursive
!pip install -q -r requirements.txt
# IMPORTANT: Save the ai-toolkit path for subprocesses
AI_TOOLKIT_DIR = '/kaggle/working/ai-toolkit'

# Force restart instruction for user
print('\n' + '='*50)
print('IMPORTANT: If you see a NumPy error after this, please RESTART')
print('the session (Run -> Restart Session) and run again from Step 2.')
print('='*50)

In [ ]:
# Step 2: Prepare Dataset from HF
import os
import random
from datasets import load_dataset
from tqdm import tqdm
from huggingface_hub import login
from PIL import Image

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except:
    HF_TOKEN = ''

if HF_TOKEN:
    login(token=HF_TOKEN)

DATASET_ID = 'darklord8777/sprites'
SAVE_DIR = '/kaggle/working/train_data'
os.makedirs(SAVE_DIR, exist_ok=True)

ds = load_dataset(DATASET_ID, split='train')
print("Dataset Features:", ds.features)
print(f"Total images available: {len(ds)}")

# Settings for image collection
SMOKE_TEST = True # Set to False for the full run
LIMIT = 20 if SMOKE_TEST else len(ds)

print(f'Processing {LIMIT} images...')

count = 0
for i, item in enumerate(tqdm(ds)):
    if count >= LIMIT: break
    
    try:
        img = item['image'].convert('RGB')
        img_path = os.path.join(SAVE_DIR, f'sprite_{i:04d}.png')
        txt_path = os.path.join(SAVE_DIR, f'sprite_{i:04d}.txt')
        
        img.save(img_path)
        
        # Improved diversity in caption generation
        cls_val = item.get('class', 'character')
        act_val = item.get('action', 'idle')
        view_dir = item.get('direction', 'front') # Renamed to view_dir to avoid shadowing dir()
        
        # Varying phrasing to help generalization
        templates = [
            f'a pixel art sprite of a {cls_val}, {act_val} pose, {view_dir} view, high quality, transparent background',
            f'pixel art style {cls_val} performing {act_val}, seen from the {view_dir}, clean edges, sprite sheet style',
            f'a {act_val} {cls_val} sprite, {view_dir} perspective, pixelated, video game asset'
        ]
        caption = random.choice(templates)
        
        with open(txt_path, 'w') as f:
            f.write(caption)
        count += 1
    except Exception as e:
        print(f"Skipping image {i} due to error: {e}")

# Sanity check counts
imgs = [f for f in os.listdir(SAVE_DIR) if f.endswith('.png')]
txts = [f for f in os.listdir(SAVE_DIR) if f.endswith('.txt')]
print(f"Sanity Check: {len(imgs)} images, {len(txts)} captions found.")
if len(imgs) != len(txts):
    print("WARNING: Mismatch between image and caption counts!")

In [ ]:

import yaml

# NOTE: FLUX.1-dev (12B) does NOT fit on free Kaggle T4 (15GB) - OOMs at model load.
# SD 1.5 (~1GB fp16) trains comfortably with high rank + resolution.
# For FLUX, use a paid GPU (A100 40GB+). Swap MODEL_ID below if you have one.

MODEL_ID = 'runwayml/stable-diffusion-v1-5'  # fits T4 free tier
# MODEL_ID = 'black-forest-labs/FLUX.1-dev'  # only if you have >=24GB VRAM

is_flux = 'flux' in MODEL_ID.lower()

def create_config(name, steps, push_to_hub=False, sample_every=250, save_every=500):
    cfg = {
        'job': 'extension',
        'config': {
            'name': name,
            'process': [{
                'type': 'sd_trainer',
                'training_folder': '/kaggle/working/output',
                'device': 'cuda:0',
                'model': {
                    'name_or_path': MODEL_ID,
                    'is_flux': is_flux,
                    'quantize': is_flux,  # only NF4 for FLUX
                },
                'network': {
                    'type': 'lora',
                    'linear': 16 if not is_flux else 4,
                    'linear_alpha': 16 if not is_flux else 4,
                },
                'train': {
                    'batch_size': 1,
                    'gradient_accumulation_steps': 4,
                    'gradient_checkpointing': True,
                    'steps': steps,
                    'lr': 0.0001,
                    'optimizer': 'adamw8bit',
                    'dtype': 'bf16',
                },
                'datasets': [{
                    'folder_path': '/kaggle/working/train_data',
                    'caption_ext': 'txt',
                    'resolution': [512 if not is_flux else 384],
                }],
                'save': {
                    'push_to_hub': push_to_hub,
                    'hf_repo_id': 'darklord8777/sprite-generator-model',
                    'hf_private': True,
                    'save_every': save_every,
                    'max_step_saves_to_keep': 3,
                },
                'sample': {
                    'sampler': 'ddim' if not is_flux else 'flowmatch',
                    'neg': '',
                    'sample_every': sample_every,
                    'seed': 42,
                    'prompts': [
                        'a pixel art sprite of a character, idle pose, front view',
                        'a pixel art sprite of an enemy, attack pose, side view',
                    ],
                },
            }],
        },
    }
    return cfg

with open('/kaggle/working/smoke_test.yaml', 'w') as f:
    yaml.dump(create_config('sprite_smoke_test', 300, push_to_hub=False), f)

with open('/kaggle/working/full_run.yaml', 'w') as f:
    yaml.dump(create_config('sprite_lora_full', 2000, push_to_hub=True, save_every=250), f)

print(f'Configs generated for {MODEL_ID} (is_flux={is_flux})')

with open('/kaggle/working/smoke_test.yaml') as f:
    raw = yaml.safe_load(f)
    prompts = raw['config']['process'][0]['sample']['prompts']
    for i, p in enumerate(prompts):
        assert isinstance(p, str), f'Prompt {i} is {type(p).__name__}, expected str!'
    print('YAML prompt format: OK (plain strings)')




In [ ]:
# Step 4: Run Training
import os, torch, gc
os.environ['HF_TOKEN'] = HF_TOKEN

# Clear GPU memory before starting
gc.collect()
torch.cuda.empty_cache()

# RECOMMENDED: Run smoke_test.yaml first!
# Must run from ai-toolkit dir so local imports resolve correctly
!cd /kaggle/working/ai-toolkit && python run.py /kaggle/working/smoke_test.yaml

# ONCE VERIFIED, uncomment below for the full run:
# !cd /kaggle/working/ai-toolkit && python run.py /kaggle/working/full_run.yaml

## Step 5: Validation & Inference
After training, run this cell to test the trained LoRA.

In [ ]:
from diffusers import StableDiffusionPipeline, FluxPipeline
from PIL import Image
import numpy as np
import torch
import os
from glob import glob

def assess_sprite(img):
    """Quality metrics for pixel art: blur, palette size, edge crispness."""
    arr = np.array(img.convert('RGB'))
    gray = np.array(img.convert('L'), dtype=np.float32)
    h, w = gray.shape

    # 1. Blur: Laplacian variance (higher = sharper)
    lap_kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
    from scipy.ndimage import convolve
    lap_out = convolve(gray, lap_kernel, mode='constant')
    blur_score = float(np.var(lap_out))

    # 2. Unique colors in non-transparent area
    if arr.shape[2] == 4:
        mask = arr[:, :, 3] > 128
        if mask.sum() > 0:
            colors = arr[mask][:, :3]
        else:
            colors = arr[:, :, :3].reshape(-1, 3)
    else:
        colors = arr.reshape(-1, 3)
    palette = len(set(tuple(c) for c in colors))

    # 3. Edge crispness: mean horizontal difference
    h_edge = np.abs(np.diff(gray, axis=1)).mean()
    v_edge = np.abs(np.diff(gray, axis=0)).mean()
    edge_score = float((h_edge + v_edge) / 2)

    return blur_score, palette, edge_score

def quality_report(blur, palette, edge):
    """Print readable quality assessment."""
    print(f"  Sharpness (Laplacian var): {blur:.1f}  {'✓ sharp' if blur > 50 else '⚠ blurry' if blur < 20 else '~ okay'}")
    print(f"  Palette colors: {palette}  {'✓ clean' if palette < 64 else '⚠ noisy' if palette > 128 else '~ okay'}")
    print(f"  Edge contrast: {edge:.1f}  {'✓ crisp' if edge > 20 else '⚠ soft' if edge < 8 else '~ okay'}")
    if blur > 50 and palette < 64 and edge > 20:
        print(f"  → RECOMMENDED: NEAREST upscale (clean pixel art)")
    elif blur < 20 or palette > 128:
        print(f"  → RECOMMENDED: no upscale or Real-ESRGAN clean-up first")
    else:
        print(f"  → RECOMMENDED: NEAREST upscale (acceptable quality)")

# Find the latest LoRA weights
lora_paths = glob("/kaggle/working/output/*sprite*/**/*.safetensors", recursive=True)

if lora_paths:
    latest_lora = sorted(lora_paths, key=os.path.getmtime)[-1]
    print(f"Loading LoRA: {latest_lora}")
    
    # Auto-detect which model to use
    try:
        import yaml
        with open('/kaggle/working/smoke_test.yaml') as f:
            cfg = yaml.safe_load(f)
        model_id = cfg['config']['process'][0]['model']['name_or_path']
        is_flux = cfg['config']['process'][0]['model'].get('is_flux', False)
    except:
        model_id = 'runwayml/stable-diffusion-v1-5'
        is_flux = False
    
    print(f'Using model: {model_id}')
    pipe_cls = FluxPipeline if is_flux else StableDiffusionPipeline
    pipe_kwargs = {'torch_dtype': torch.bfloat16} if is_flux else {'torch_dtype': torch.float16, 'safety_checker': None}
    pipe = pipe_cls.from_pretrained(model_id, **pipe_kwargs)
    pipe.load_lora_weights(latest_lora)
    pipe.to("cuda")

    prompts = [
        "a pixel art sprite of a character, idle pose, front view, high quality, transparent background", # On-template
        "a pixel art dragon, flying pose, side view, fiery breath, pixel art style" # Off-template
    ]
    gen_kwargs = {'num_inference_steps': 28, 'guidance_scale': 7.0, 'width': 512, 'height': 512}
    if is_flux:
        gen_kwargs['guidance_scale'] = 3.5

    for i, prompt in enumerate(prompts):
        image = pipe(prompt, **gen_kwargs).images[0]
        print(f"\n--- Prompt {i}: {prompt[:50]}... ---")
        blur, palette, edge = assess_sprite(image)
        quality_report(blur, palette, edge)
        # Auto-upscale based on quality
        if blur > 50 and palette < 64 and edge > 20:
            upscale_by = 4  # clean → 4x
        elif blur > 20 and palette < 128:
            upscale_by = 2  # okay → 2x
        else:
            upscale_by = 1  # noisy → keep as-is
        if upscale_by > 1:
            w, h = image.size
            image = image.resize((w * upscale_by, h * upscale_by), Image.NEAREST)
            print(f"  → Auto-upscaled {upscale_by}x ({w}x{h} → {w*upscale_by}x{h*upscale_by})")
        image.save(f"/kaggle/working/val_{i}.png")
        display(image)
else:
    print("No LoRA weights found. Complete training first.")